In [1]:
import os
import sys
from pathlib import Path
import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import RobustScaler
import json
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
import gc

# Ensure the project root is on sys.path so local utils/ is imported instead of an installed package
cur_dir = Path(os.getcwd())
proj_dir = cur_dir.resolve().parent.parent
sys.path.insert(0, str(proj_dir))

from utils.preprocessing import *

In [2]:
cur_dir = Path(os.getcwd())
proj_dir = cur_dir.resolve().parent.parent
RAW_DATA_PATH = proj_dir / "data" / "raw" / "files"
OUTPUT_DATA_PATH = proj_dir / "data" / "preprocessed"

In [3]:
EXCLUDE_FILES = ["chb12_27", "chb12_28", "chb12_29"]

SFREQ = 256
WINDOW_SEC = 5
OVERLAP = 0.5

## Loading data

In [4]:
files = load_files(base_path=RAW_DATA_PATH, exclude_files=EXCLUDE_FILES)


Elaborating patient chb13

Finish to elaborate patient chb13

Elaborating patient chb04

Finish to elaborate patient chb04

Elaborating patient chb19

Finish to elaborate patient chb19

Elaborating patient chb12

Skipping file chb12_27

Skipping file chb12_28

Skipping file chb12_29

Finish to elaborate patient chb12


In [5]:
files.keys()

dict_keys(['chb04_01', 'chb04_02', 'chb04_02_99perc', 'chb04_05', 'chb04_08', 'chb12_06', 'chb12_08', 'chb12_09', 'chb12_10', 'chb12_11', 'chb12_23', 'chb12_33', 'chb12_36', 'chb12_38', 'chb12_42'])

## Elaboration and cleaning of the data

The filterbank spans the frequency range 0.5-25 Hz since most seizure and nonseizure EEG activity falls within this range.

In [6]:
for name, file in files.items():
    print(f"\n|--- Elaborating patient {name} ---|")
    
    # Extrapolate channels name
    ch_names = file.ch_names
    
    # Apply band-pass filter
    file.filter(0.5, 25, verbose=False)
    
    # Extrapolate filtered data and times
    data_filtered = file.get_data()
    
    del file
    
    # Generate scaler
    scaler = RobustScaler()

    # Cleaning of missing channels
    data_ch_filt = data_dict(data_filtered, ch_names, scaler)
    
    # Computing the windows and overlapping size
    win_size = int(WINDOW_SEC*SFREQ)
    step = int(win_size*(1-OVERLAP))

    # retrieve data from each channel
    chs = list(data_ch_filt.keys())
    signals = np.array([data_ch_filt[ch] for ch in chs])
    
    del data_ch_filt
    
    print(f"\nShape of the signal: {signals.shape}")
    
    windows = []

    # Sampling the windows
    for start in range(0, signals.shape[1]-win_size, step):     
        window = signals[:, start:start+win_size]
        windows.append(window)
        
    del signals

    windows = np.array(windows)
    
    print(f"\nShape of the windows: {windows.shape}")
    
    base_name = name.split('_')[0]
    
    if not os.path.exists(os.path.join(OUTPUT_DATA_PATH, base_name)):
        os.makedirs(os.path.join(OUTPUT_DATA_PATH, base_name))
    
    target_path = OUTPUT_DATA_PATH / f"{name.split('_')[0]}" / f'{name}.npz'
    
    # save the file name
    np.savez(target_path, windows=windows, channels=chs)
    
    del windows
    gc.collect()
    
    print(f"\n|--- Finish elaborating patient {name} ---|")
    
print(f"\nFinish to elaborate all the files")


|--- Elaborating patient chb04_01 ---|

Shape of the signal: (21, 3685888)

Shape of the windows: (5758, 21, 1280)

|--- Finish elaborating patient chb04_01 ---|

|--- Elaborating patient chb04_02 ---|

Shape of the signal: (21, 3686400)

Shape of the windows: (5758, 21, 1280)

|--- Finish elaborating patient chb04_02 ---|

|--- Elaborating patient chb04_02_99perc ---|

Shape of the signal: (21, 3646976)

Shape of the windows: (5697, 21, 1280)

|--- Finish elaborating patient chb04_02_99perc ---|

|--- Elaborating patient chb04_05 ---|

Shape of the signal: (21, 2441216)

Shape of the windows: (3813, 21, 1280)

|--- Finish elaborating patient chb04_05 ---|

|--- Elaborating patient chb04_08 ---|

Shape of the signal: (22, 3686400)

Shape of the windows: (5758, 22, 1280)

|--- Finish elaborating patient chb04_08 ---|

|--- Elaborating patient chb12_06 ---|

Shape of the signal: (26, 923136)

Shape of the windows: (1441, 26, 1280)

|--- Finish elaborating patient chb12_06 ---|

|--- Ela